[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [SQLModel, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/sqlmodel-deep-dive.html)

# Create, Read and Update Models


## What you will be able to do

Write the family of models a stored thing needs: a base with the shared fields, the table model, one
model for what a caller may send, one for what may be sent back, and one for a change. Build a row
from a create model and a public answer from a row, both with `model_validate`. Apply a partial
change with `sqlmodel_update` and know exactly why it takes `model_dump(exclude_unset=True)`.
Recognize the failures: a secret in an answer, an age quietly emptied, a required column set to
nothing, and a change that went nowhere.


## The idea

### The problem

One class can be a table and a Pydantic model at once, and that is what makes SQLModel worth using.
It does not follow that one class can be every model a program needs.

A `Hero` row holds a secret name. Nothing that leaves the program may carry it, and the class that
describes the row carries it by definition. Hand a `Hero` to anything that serializes, and the secret
goes with it, with no error and nothing in the output to say something went wrong: the answer simply
has a field in it that nobody meant to publish.

Changes have the opposite problem. A change usually names a few fields, and the object it is applied
to has all of them. Build a dictionary of every field, with `None` where the caller said nothing, and
applying it sets those fields to `None`: an age is emptied and nothing says so, a required column is
emptied and the database refuses the whole update. Both come from one missing idea, which is the
difference between a field set to nothing and a field not mentioned.

### What the family is

> A **base model** holds the fields every version shares and has no table. The **table model**
> inherits it, adds the `id` and whatever must never leave, and is the only one with `table=True`.
> A **create model** inherits the base and adds what a caller must supply, with no `id`. A **public
> model** inherits the base and adds the `id`, and leaves out the secrets. An **update model** has
> every field optional, because anything it does not mention must be left alone.
> **`Model.model_validate(other)`** converts between them, reading fields off a dictionary or an
> object, and **`row.sqlmodel_update(changes)`** applies a dictionary of changes to a row.

### Why it works that way

- **Inheritance writes the shared fields once.** A field added to the base appears in the table, in
  what is accepted and in what is sent back, and a field that belongs to one of them is written
  there and nowhere else.
- **What may be sent in is not what may be sent out.** A caller may not choose an id and must supply
  a secret; a reader may see the id and must not see the secret. Those are different lists, so they
  are different classes.
- **An update model is all optional on purpose.** `None` in it means the caller wrote `None`, and a
  field missing from it means the caller said nothing, and `model_dump(exclude_unset=True)` is what
  tells the two apart.
- **`model_validate` reads objects as well as dictionaries.** A model with no table takes its values
  from an object's attributes, so turning a row into a public model is one call.
- **The table model stays the only one with a table.** The others exist for as long as a call takes.

### Where this shows up

Every service that stores what it is sent. The names here are the ones SQLModel's own documentation
uses, and the **SQLModel in FastAPI** notebook hands each of them to a route, where the create model
becomes the body a route accepts and the public model becomes what it answers with. The
**Validation and table=True** notebook is why the create model exists at all: it is the one whose
constructor checks.

### What this notebook covers

- One class doing three jobs, and what leaks
- A change built from every field, and what it empties
- The family: base, table, create, public and update
- Making a row from a create model, and an answer from a row
- Applying a change without emptying anything
- Which model to reach for
- A hero created, read and changed, finished
- Four failures, from a secret in an answer to a change that went nowhere

### A first look

Before any of the detail, here is the whole idea in a few lines. There is nothing to run yet: read
it, and read the output underneath it. Everything from Setup onward is where you start running
things, and the rest of the notebook takes this apart piece by piece.

```python
from sqlmodel import Field, SQLModel


class HeroBase(SQLModel):
    name: str
    age: int | None = None


class Hero(HeroBase, table=True):
    id: int | None = Field(default=None, primary_key=True)
    secret_name: str


class HeroPublic(HeroBase):
    id: int


hero = Hero(id=1, name="Deadpond", age=30, secret_name="Dive Wilson")
print("the row   :", hero.model_dump())
print("the answer:", HeroPublic.model_validate(hero).model_dump())
```

```
the row   : {'name': 'Deadpond', 'age': 30, 'id': 1, 'secret_name': 'Dive Wilson'}
the answer: {'name': 'Deadpond', 'age': 30, 'id': 1}
```

Two classes from one base, and the difference between them is the secret name. The row has it
because the table has it; the answer does not have it because `HeroPublic` never declared it, and
`model_validate` copies the fields the model it is building has, not the fields the object it was
given has.


## Setup

Thirteen imports, one of them installed first where it is missing, the cast, three helpers, the
starting classes, the engine, and the database built and loaded.

- `sqlmodel` is the library, and `SQLModel`, `Field`, `Session`, `create_engine` and `select`, from
  it, are the classes, the session and the reading. Colab does not have SQLModel, so the cell
  installs 0.0.42 with `pip` where it is missing, and `version` and `PackageNotFoundError`, from
  `importlib.metadata`, `subprocess` and `sys` find out whether it is
- `ValidationError`, from `pydantic`, and `IntegrityError`, from `sqlalchemy.exc`, are what the
  Common errors catch
- `event` and `insert`, from `sqlalchemy`, are the pragma on every connection and the rows `build`
  loads without a session
- `re` takes memory addresses out of a message, and `warnings` and `contextlib` make `catching`,
  which collects the warning that a class defined a second time raises
- `Path` names the database file, and `shutil` removes the scratch folder at the start and at the end
- `TEAMS` and `HEROES` are the cast, which `build` loads

The classes here are the plain `Hero` and `Team` of the earlier notebooks, because the first two
worked examples are about what goes wrong with only those. The family is written in the third, which
empties `SQLModel.metadata` and defines the models again. The tables do not change, so the database
built here is the one the rest of the notebook reads.


In [1]:
import contextlib
import re
import shutil
import subprocess
import sys
import warnings
from importlib.metadata import PackageNotFoundError, version
from pathlib import Path

try:
    version("sqlmodel")
except PackageNotFoundError:                                        # Colab has no SQLModel: install the pinned version
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--root-user-action=ignore",
                    "sqlmodel==0.0.42"], check=True)

import sqlmodel
from pydantic import ValidationError
from sqlalchemy import event, insert
from sqlalchemy.exc import IntegrityError
from sqlmodel import Field, Session, SQLModel, create_engine, select

TEAMS = [                                                           # name, headquarters
    ("Preventers", "Sharp Tower"),
    ("Z-Force", "Sister Margaret's Bar"),
    ("Wakaland Guard", "Grand Palace"),                             # no heroes, for the joins that keep a team anyway
]

HEROES = [                                                          # name, secret name, age, team
    ("Deadpond", "Dive Wilson", None, "Z-Force"),
    ("Spider-Boy", "Pedro Parqueador", 16, "Preventers"),
    ("Rusty-Man", "Tommy Sharp", 48, "Preventers"),
    ("Tarantula", "Natalia Roman-on", 32, "Preventers"),
    ("Black Lion", "Trevor Challa", 35, "Z-Force"),
    ("Dr. Weird", "Steve Weird", 36, "Z-Force"),
    ("Captain North America", "Esteban Rogelios", 93, "Preventers"),
    ("Princess Sure-E", "Sure-E", None, None),                      # on no team
]


def message(error):
    """An error's text, without the memory address or the version link that make no two runs agree."""
    text = re.sub(r"0x[0-9a-f]+", "0x...", str(error))
    return "\n".join(line for line in text.splitlines() if "errors.pydantic.dev" not in line).strip()


def fields(model):
    """A model's values in the order its class declares them, which a loaded object does not keep."""
    return {name: getattr(model, name) for name in type(model).model_fields}


@contextlib.contextmanager
def catching():
    """Collects warnings instead of printing them: Python prints one with the file that raised it."""
    with warnings.catch_warnings(record=True) as raised:
        warnings.simplefilter("always")
        yield raised


class Team(SQLModel, table=True):
    id: int | None = Field(default=None, primary_key=True)
    name: str = Field(index=True, max_length=50)
    headquarters: str = Field(max_length=60)


class Hero(SQLModel, table=True):
    id: int | None = Field(default=None, primary_key=True)
    name: str = Field(index=True, max_length=50)
    secret_name: str = Field(max_length=60)
    age: int | None = Field(default=None, index=True)
    team_id: int | None = Field(default=None, foreign_key="team.id")


def hero_engine(path=None, echo=False):
    """The guide's engine: the database in a file, or with no path one in memory, with foreign keys checked."""
    engine = create_engine("sqlite://" if path is None else f"sqlite:///{path}", echo=echo)

    @event.listens_for(engine, "connect")
    def enforce_foreign_keys(connection, record):
        cursor = connection.cursor()
        cursor.execute("PRAGMA foreign_keys=ON")
        cursor.close()

    return engine


def build(engine):
    """Create the tables and load the cast, without a session: every notebook starts from the same rows."""
    SQLModel.metadata.create_all(engine)
    with engine.begin() as connection:
        connection.execute(insert(Team), [{"name": name, "headquarters": where} for name, where in TEAMS])
        teams = {name: number for number, (name, _) in enumerate(TEAMS, start=1)}
        connection.execute(insert(Hero), [{"name": name, "secret_name": secret, "age": age,
                                           "team_id": teams.get(team)}
                                          for name, secret, age, team in HEROES])

shutil.rmtree("scratch", ignore_errors=True)                        # a rerun starts from the same eight heroes
Path("scratch").mkdir()
engine = hero_engine("scratch/heroes.db")
build(engine)

with Session(engine) as session:
    print("sqlmodel", sqlmodel.__version__, "|", len(session.exec(select(Hero)).all()), "heroes loaded")


sqlmodel 0.0.42 | 8 heroes loaded


## Worked examples

### One class doing three jobs, and what leaks

A function that reads a hero and gives it back, which is as natural as code gets:


In [2]:
def hero_for_the_page(session, hero_id):
    """One hero, ready to be shown."""
    return session.get(Hero, hero_id)


with Session(engine) as session:
    shown = hero_for_the_page(session, 1)
    print("what the caller gets:", fields(shown))
    print("the fields in a dump:", sorted(shown.model_dump()))


what the caller gets: {'id': 1, 'name': 'Deadpond', 'secret_name': 'Dive Wilson', 'age': None, 'team_id': 2}
the fields in a dump: ['age', 'id', 'name', 'secret_name', 'team_id']


The secret name is in the object, and in the list of fields anything that serializes it will write.
Nothing raised, nothing warned, and every caller of that function now has a value that exists to be
kept. The class is the table, the table has a secret column, and anything
that turns the class into a dictionary includes it.

Leaving the field out by hand is not the answer either. It puts the list of allowed fields in every
function that returns a hero, which is the same list written many times over, and a new secret column
is then a bug in every one of those functions until somebody remembers them all.

### A change built from every field, and what it empties

A hero changes team. The caller sends the fields it knows about, and the function applies them:


In [3]:
def change_hero(session, hero_id, changes):
    """Apply a dictionary of changes to a hero."""
    hero = session.get(Hero, hero_id)
    hero.sqlmodel_update(changes)
    session.add(hero)
    session.commit()
    session.refresh(hero)
    return hero


with Session(engine) as session:
    print("before:", fields(session.get(Hero, 3)))
    changed = change_hero(session, 3, {"name": "Rusty-Man", "secret_name": "Tommy Sharp",
                                       "age": None, "team_id": 2})
    print("after :", fields(changed))


before: {'id': 3, 'name': 'Rusty-Man', 'secret_name': 'Tommy Sharp', 'age': 48, 'team_id': 1}
after : {'id': 3, 'name': 'Rusty-Man', 'secret_name': 'Tommy Sharp', 'age': None, 'team_id': 2}


The caller meant to move Rusty-Man to team 2. What it sent was every field, with `None` for the age
because its form had nothing in that box, and the age was 48 and is now empty. No error, no warning,
and the only way to notice is to look at the row.

The same mistake against a column that cannot be empty is louder, and the last of the Common errors
is where that one lands. Both have one cause: a dictionary that says `None` where the caller meant to
say nothing at all.

### The family: base, table, create, public and update

Five classes, one of them with a table. `SQLModel.metadata.clear()` comes first because `hero` and
`team` are already in the metadata from Setup, and the classes are about to be written again:


In [4]:
with catching() as warned:                                          # the class names are used a second time
    SQLModel.metadata.clear()

    class HeroBase(SQLModel):
        """What every hero has, wherever it is going. No table."""

        name: str = Field(index=True, max_length=50)
        age: int | None = Field(default=None, index=True)
        team_id: int | None = Field(default=None, foreign_key="team.id")


    class Hero(HeroBase, table=True):
        """The row: the shared fields, the id the database gives, and the secret."""

        id: int | None = Field(default=None, primary_key=True)
        secret_name: str = Field(max_length=60)


    class HeroCreate(HeroBase):
        """What a caller may send to make a hero: no id, and the secret it must supply."""

        secret_name: str = Field(max_length=60)


    class HeroPublic(HeroBase):
        """What may be sent back: the shared fields and the id, and no secret at all."""

        id: int


    class HeroUpdate(SQLModel):
        """A change: every field optional, so that anything left out means leave it alone."""

        name: str | None = None
        age: int | None = None
        team_id: int | None = None
        secret_name: str | None = None


    class Team(SQLModel, table=True):
        id: int | None = Field(default=None, primary_key=True)
        name: str = Field(index=True, max_length=50)
        headquarters: str = Field(max_length=60)


SQLModel.metadata.create_all(engine)                                # the tables are already there
print("tables in the metadata:", sorted(SQLModel.metadata.tables))
for model in (HeroBase, Hero, HeroCreate, HeroPublic, HeroUpdate):
    print(f"  {model.__name__:<11} table {str(hasattr(model, '__table__')):<5} fields {list(model.model_fields)}")


tables in the metadata: ['hero', 'team']
  HeroBase    table False fields ['name', 'age', 'team_id']
  Hero        table True  fields ['name', 'age', 'team_id', 'id', 'secret_name']
  HeroCreate  table False fields ['name', 'age', 'team_id', 'secret_name']
  HeroPublic  table False fields ['name', 'age', 'team_id', 'id']
  HeroUpdate  table False fields ['name', 'age', 'team_id', 'secret_name']


`HeroBase` has the three fields every version shares and no table. `Hero` adds the id and the secret,
and is the only one `create_all` builds. `HeroCreate` adds the secret a caller must supply and has no
id, so a caller cannot choose one. `HeroPublic` adds the id and never mentions the secret.
`HeroUpdate` shares nothing: every field is optional, including the ones that are required
everywhere else, because a change that does not mention a field must leave it alone.

The tables did not change, so the rows loaded in Setup are still there and the new `Hero` reads them.

### Making a row from a create model, and an answer from a row

`model_validate` goes both ways, and neither direction needs a field list written by hand:


In [5]:
arriving = HeroCreate(name="Ghost Girl", secret_name="Ana Vega", age="27", team_id=1)
print("checked on the way in:", fields(arriving))

hero = Hero.model_validate(arriving)
with Session(engine, expire_on_commit=False) as session:
    session.add(hero)
    session.commit()

answer = HeroPublic.model_validate(hero)
print("the row              :", fields(hero))
print("what may be sent back:", answer.model_dump())


checked on the way in: {'name': 'Ghost Girl', 'age': 27, 'team_id': 1, 'secret_name': 'Ana Vega'}
the row              : {'name': 'Ghost Girl', 'age': 27, 'team_id': 1, 'id': 9, 'secret_name': 'Ana Vega'}
what may be sent back: {'name': 'Ghost Girl', 'age': 27, 'team_id': 1, 'id': 9}


The age arrived as the text `"27"` and `HeroCreate`, which has no table, converted it and would have
refused anything it could not convert. `Hero.model_validate(arriving)` built the row from the model,
and `HeroPublic.model_validate(hero)` built the answer from the row, reading the fields off the
object. The secret name is in the row and not in the answer, and no function had to remember that.

### Applying a change without emptying anything

`HeroUpdate` is where a change is checked, and `exclude_unset=True` is what separates a field the
caller set to nothing from a field the caller never mentioned:


In [6]:
change = HeroUpdate(age=48)                                         # one field mentioned, three not

print("every field      :", change.model_dump())
print("only what was set:", change.model_dump(exclude_unset=True))

with Session(engine) as session:
    rusty = session.get(Hero, 3)
    print("before:", fields(rusty))
    rusty.sqlmodel_update(change.model_dump(exclude_unset=True))
    session.add(rusty)
    session.commit()
    session.refresh(rusty)
    print("after :", fields(rusty))


every field      : {'name': None, 'age': 48, 'team_id': None, 'secret_name': None}
only what was set: {'age': 48}
before: {'name': 'Rusty-Man', 'age': None, 'team_id': 2, 'id': 3, 'secret_name': 'Tommy Sharp'}
after : {'name': 'Rusty-Man', 'age': 48, 'team_id': 2, 'id': 3, 'secret_name': 'Tommy Sharp'}


The full dump has all four fields, three of them `None`, which is what emptied the age in the second
worked example. `exclude_unset=True` gives the one field the caller actually set, `sqlmodel_update`
applies that one, and the age that was emptied earlier is back with the name, the team and the secret
name all untouched.

A caller that really means to empty a field sends it as `None`, and then it is set:
`HeroUpdate(age=None)` has `age` in `model_dump(exclude_unset=True)` because it was set, to `None`.

### Which model to reach for

| The model | Has a table | Carries | Used for |
|---|---|---|---|
| `HeroBase` | no | the shared fields | inheriting, and nothing else |
| `Hero` | yes | everything, id and secret included | rows, and only rows |
| `HeroCreate` | no | the shared fields and the secret | checking what a caller sends to create one |
| `HeroPublic` | no | the shared fields and the id | what may be sent back |
| `HeroUpdate` | no | every field, all optional | a change, with `exclude_unset=True` |

Two more shapes appear as a program grows, and both follow the same rule: a `HeroCreate` with a
password is matched by a table model with a hashed one, so the field the caller sends is not the
field that is stored; and a list answer is usually its own model, because what belongs in a list is
rarely everything that belongs on a page.

### A hero created, read and changed, finished

The pieces of this notebook in three functions, which are the three things a stored thing needs:


In [7]:
def create_hero(engine, raw):
    """Check what arrived, write the row, and give back what may be shown."""
    arriving = HeroCreate.model_validate(raw)
    hero = Hero.model_validate(arriving)
    with Session(engine, expire_on_commit=False) as session:
        session.add(hero)
        session.commit()
    return HeroPublic.model_validate(hero)


def read_hero(engine, hero_id):
    """One hero, as it may be shown, or None."""
    with Session(engine) as session:
        hero = session.get(Hero, hero_id)
        return HeroPublic.model_validate(hero) if hero else None


def change_hero(engine, hero_id, raw):
    """Apply only the fields the change mentions, and give back what may be shown."""
    change = HeroUpdate.model_validate(raw)
    with Session(engine, expire_on_commit=False) as session:
        hero = session.get(Hero, hero_id)
        if hero is None:
            return None
        hero.sqlmodel_update(change.model_dump(exclude_unset=True))
        session.add(hero)
        session.commit()
    return HeroPublic.model_validate(hero)


made = create_hero(engine, {"name": "Night Owl", "secret_name": "Bruce Kent", "age": "41"})
print("created:", made.model_dump())
print("read   :", read_hero(engine, made.id).model_dump())
print("changed:", change_hero(engine, made.id, {"team_id": 2}).model_dump())
print("missing:", read_hero(engine, 999))


created: {'name': 'Night Owl', 'age': 41, 'team_id': None, 'id': 10}
read   : {'name': 'Night Owl', 'age': 41, 'team_id': None, 'id': 10}
changed: {'name': 'Night Owl', 'age': 41, 'team_id': 2, 'id': 10}
missing: None


Three functions, one table model, and not one of them writes out a list of fields. The secret name
went in with the creation and is in none of the three answers. The change mentioned one field and
left the rest, including the age that came in as text and is a number in the row.

### Where each part came from

| In the three functions | What it relies on | The section that showed it |
|---|---|---|
| `HeroCreate.model_validate(raw)` | a model with no table, whose constructor checks | the **Validation and table=True** notebook |
| `Hero.model_validate(arriving)` | one model built from another | Making a row from a create model |
| `HeroPublic.model_validate(hero)` | a model built from an object, carrying only its own fields | Making a row from a create model |
| `change.model_dump(exclude_unset=True)` | the fields the caller set, and no others | Applying a change without emptying anything |
| `expire_on_commit=False` | an object still readable after its session closed | the **Sessions** notebook |


## Your turn

Six tasks. Write your answer in the cell under each task and run it.

Try a task before you look at its answer. Reading a solution teaches you much less than getting
there yourself, even slowly.

When you are ready: [**open the solutions notebook**](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/sqlmodel-deep-dive/07-create-read-and-update-models-solutions.ipynb).

**1.** Write `TeamBase`, `TeamCreate` and `TeamPublic` for the `Team` table model, with the
headquarters shared and the id only in the public one, and print the fields of each.


In [8]:
# your code here


**2.** Create a team from a `TeamCreate` and print it as a `TeamPublic`, without naming a single
field by hand.


In [9]:
# your code here


**3.** Write `TeamUpdate` with every field optional, and print `model_dump()` and
`model_dump(exclude_unset=True)` for a change that names only the headquarters.


In [10]:
# your code here


**4.** Apply that change to a team and show the name is untouched. Then apply the full dump to
another team and show what it did.


In [11]:
# your code here


**5.** Write `HeroListed`, a model with no table carrying the id and the name alone, and turn every
hero of team 2 into one.


In [12]:
# your code here


**6.** Write `create_team(engine, raw)`, which checks, writes and returns the public model, and show
what it does with a headquarters that is missing.


In [13]:
# your code here


## Common errors

### ValidationError: 1 validation error for HeroPublic


In [14]:
fresh = Hero.model_validate(HeroCreate(name="Stone Hand", secret_name="Marco Pietra", age=38))
print("id before the commit:", fresh.id)
HeroPublic.model_validate(fresh)


id before the commit: None


ValidationError: 1 validation error for HeroPublic
id
  Input should be a valid integer [type=int_type, input_value=None, input_type=NoneType]
    For further information visit https://errors.pydantic.dev/2.12/v/int_type

`HeroPublic` requires an `id`, since anything sent back has been stored and a stored hero has one.
This hero has not been written yet, so its id is `None`, and `None` is not an integer. The message
says exactly that, and the fix is to build the public model after the commit, which every worked
example above does.

The same error with a different cause is a hero that was committed and then expired: read the id
before the session closes, or keep `expire_on_commit=False`, both of which the **Sessions** notebook
covers.

### sqlalchemy.exc.IntegrityError: (sqlite3.IntegrityError) NOT NULL constraint failed: hero.name


In [15]:
with Session(engine) as session:
    hero = session.get(Hero, 5)
    hero.sqlmodel_update(HeroUpdate(age=36).model_dump())           # every field, three of them None
    session.commit()


IntegrityError: (sqlite3.IntegrityError) NOT NULL constraint failed: hero.name
[SQL: UPDATE hero SET name=?, age=?, team_id=?, secret_name=? WHERE hero.id = ?]
[parameters: (None, 36, None, None, 5)]
(Background on this error at: https://sqlalche.me/e/20/gkpj)

This is the second worked example's mistake against a column that cannot be empty. The update
statement set `name`, `age`, `team_id` and `secret_name`, three of them to `None`, and the database
refused the row rather than lose the name.

Where the column accepts a null, the same statement succeeds and the value is gone, which is the
version nothing tells you about. `exclude_unset=True` is the fix for both:


In [16]:
with Session(engine) as session:
    session.rollback()
    hero = session.get(Hero, 5)
    hero.sqlmodel_update(HeroUpdate(age=36).model_dump(exclude_unset=True))
    session.add(hero)
    session.commit()
    session.refresh(hero)
    print("only the age changed:", fields(hero))


only the age changed: {'name': 'Black Lion', 'age': 36, 'team_id': 2, 'id': 5, 'secret_name': 'Trevor Challa'}


### ValidationError: 1 validation error for HeroCreate


In [17]:
HeroCreate.model_validate({"name": "Mystery Man", "age": 44})       # no secret name


ValidationError: 1 validation error for HeroCreate
secret_name
  Field required [type=missing, input_value={'name': 'Mystery Man', 'age': 44}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.12/v/missing

`secret_name` is required by `HeroCreate` and nothing supplied it. This is the create model doing its
job: the check happens where the data arrives, with a message naming the field, rather than at a
commit somewhere else with a message naming a column.

Compare it with what the table model does with the same dictionary, which is what the
**Validation and table=True** notebook is about:


In [18]:
loose = Hero(name="Mystery Man", age=44)
print("the table model took it:", fields(loose))


the table model took it: {'name': 'Mystery Man', 'age': 44, 'team_id': None, 'id': None, 'secret_name': None}


### No error, and a change that went nowhere: a field the update model does not have


In [19]:
with Session(engine) as session:
    hero = session.get(Hero, 2)
    before = fields(hero)
    hero.sqlmodel_update({"nickname": "Spidey", "team": "Preventers"})   # neither is a field
    session.add(hero)
    session.commit()
    session.refresh(hero)
    print("before:", before)
    print("after :", fields(hero))
    print("did it gain a nickname?", hasattr(hero, "nickname"))


before: {'name': 'Spider-Boy', 'age': 16, 'team_id': 1, 'id': 2, 'secret_name': 'Pedro Parqueador'}
after : {'name': 'Spider-Boy', 'age': 16, 'team_id': 1, 'id': 2, 'secret_name': 'Pedro Parqueador'}
did it gain a nickname? False


Two keys, neither of them a field of `Hero`, and `sqlmodel_update` ignored both without a word. The
row is unchanged and the call reported nothing, which is what makes a misspelled key in a change hard
to find: `team` is not `team_id`, and the hero stayed where it was.

Validating the change first is what catches it, because an update model has a fixed list of fields:


In [20]:
change = HeroUpdate.model_validate({"team": "Preventers"})
print("what the update model kept:", change.model_dump(exclude_unset=True))


class StrictUpdate(HeroUpdate):
    model_config = {"extra": "forbid"}                              # a key that is not a field is refused


try:
    StrictUpdate.model_validate({"team": "Preventers"})
except ValidationError as error:
    print(message(error))


what the update model kept: {}
1 validation error for StrictUpdate
team
  Extra inputs are not permitted [type=extra_forbidden, input_value='Preventers', input_type=str]


Pydantic drops an unknown key rather than refusing it, so the validated change is empty and the
mistake shows as a change that did nothing rather than as a wrong value written. Where callers are
outside your control, `model_config = {"extra": "forbid"}` turns the same key into a refusal that
names it.

Last, the engine lets go of the file, and this cell removes the scratch folder with the database in
it:


In [21]:
engine.dispose()
shutil.rmtree("scratch")

print("scratch still there:", Path("scratch").exists())


scratch still there: False


## Recap

- A base model with no table holds the shared fields; the table model inherits it and adds the id and
  anything that must not leave.
- A create model is what a caller may send, with no id; a public model is what may be sent back, with
  the id and no secrets.
- `Model.model_validate(other)` converts between them, reading fields off an object as readily as off
  a dictionary, and copies only the fields the model being built declares.
- An update model has every field optional, and `model_dump(exclude_unset=True)` is what tells a
  field set to `None` from a field never mentioned.
- `row.sqlmodel_update(changes)` applies a dictionary of changes and ignores any key that is not a
  field.


## What is next

The **Relationships** notebook gives the hero and the team attributes that hold each other:
`Relationship` and `back_populates`, the warning when only one side declares it, the lazy load that
fails after the session closed, and what happens to the heroes when their team is deleted.


---

&#8592; **Previous:** [Validation and table=True](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/sqlmodel-deep-dive/06-validation-and-table-true.ipynb)  &nbsp;·&nbsp;  [SQLModel, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/sqlmodel-deep-dive.html)  &nbsp;·&nbsp;  **Next:** [Relationships](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/sqlmodel-deep-dive/08-relationships.ipynb) &#8594;
